In [1]:
SIMULATION_DATE = "2026-03-19"

from config import r, STOCKS, FILEPATH, s3, EXCHANGE
import os

def download_file():
    local_path = f"{FILEPATH}/{EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}.csv"
    if not os.path.exists(local_path):
        
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        try:
            s3.download_file(
                Bucket="cashcow",
                Key=f"{EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}.csv",
                Filename=local_path
            )
            print(f"[MAIN] Downloaded {EXCHANGE}:{STOCKS[0]} for {SIMULATION_DATE}")
        except Exception as e:
            raise Exception(f"file not found, choose another date. key={EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}.csv") from e
            

download_file()

In [2]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from simulator import get_all_tick_data
df = get_all_tick_data(SIMULATION_DATE)

from StockAnalyser import Delta_analysis
instance = Delta_analysis()
for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing ticks"):
    instance.parse(row.to_dict())

/Users/gurusai/programming/STONKS/BackTestingEngine/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Loaded 3 instruments from cache
✅ Cache is up to date, no refresh needed
[INFO] loaded data into memory for simulation


Parsing ticks: 100%|██████████| 27272/27272 [00:02<00:00, 12794.86it/s]


In [5]:
import numpy as np
import pyqtgraph as pg
from PyQt5.QtCore import QRectF

def get_active_price_bounds_local(inst):
    active = np.where((inst.aggdf_buy > 0) & (inst.aggdf_sell > 0))[0]
    if active.size == 0:
        return 0, max(0, inst.WIDTH - 1)
    return int(active[0]), int(active[-1])

# ---------- Build heatmap data ----------
rows_used = int(instance.curr_time_idx)
if rows_used <= 0:
    raise ValueError("No parsed ticks available in instance.")

min_idx, max_idx = get_active_price_bounds_local(instance)
col_slice = slice(min_idx, max_idx + 1)

lh_buy = instance.lowHigh['buy'][:rows_used, col_slice]
hl_buy = instance.highLow['buy'][:rows_used, col_slice]
lh_sell = instance.lowHigh['sell'][:rows_used, col_slice]
hl_sell = instance.highLow['sell'][:rows_used, col_slice]

valid_buy = ~np.isnan(lh_buy)
valid_sell = ~np.isnan(lh_sell)

buy_code = np.zeros_like(lh_buy, dtype=np.uint8)
sell_code = np.zeros_like(lh_sell, dtype=np.uint8)

buy_code[valid_buy] = (
    (lh_buy[valid_buy] >= 0).astype(np.uint8)
    + 2 * (hl_buy[valid_buy] >= 0).astype(np.uint8)
)
sell_code[valid_sell] = (
    (lh_sell[valid_sell] >= 0).astype(np.uint8)
    + 2 * (hl_sell[valid_sell] >= 0).astype(np.uint8)
)

heatmap_data = sell_code + 4 * buy_code
invalid = ~(valid_buy & valid_sell)
heatmap_data[invalid] = 16  # transparent

# ---------- LUT ----------
base_lut = np.array([
    [255, 0, 0, 170],       # 0: Red
    [0, 114, 178, 170],     # 1: Blue
    [230, 159, 0, 170],     # 2: Orange
    [86, 180, 233, 170],    # 3: Sky Blue
    [0, 158, 115, 170],     # 4: Bluish Green
    [204, 121, 167, 170],   # 5: Purple
    [240, 228, 66, 170],    # 6: Yellow
    [0, 0, 128, 170],       # 7: Navy
    [255, 105, 180, 170],   # 8: Pink
    [27, 94, 32, 170],      # 9: Dark Green
    [139, 69, 19, 170],     # 10: Brown
    [128, 0, 0, 170],       # 11: Maroon
    [0, 128, 128, 170],     # 12: Teal
    [245, 245, 245, 170],   # 13: Off-White
    [119, 119, 119, 170],   # 14: Gray
    [57, 255, 20, 170],     # 15: Neon Green
    [0, 0, 0, 0],           # 16: TRANSPARENT
], dtype=np.ubyte)

full_lut = np.zeros((256, 4), dtype=np.ubyte)
full_lut[:17] = base_lut

min_ltp = float(instance.base_ltp + min_idx)
max_ltp = float(instance.base_ltp + max_idx)
rect = QRectF(0, min_ltp, rows_used, max(1.0, max_ltp - min_ltp + 1.0))

def make_component_map(arr, true_code):
    out = np.zeros_like(arr, dtype=np.uint8)  # 0 => red
    valid = ~np.isnan(arr)
    out[valid] = np.where(arr[valid] >= 0, true_code, 0).astype(np.uint8)
    out[~valid] = 16  # transparent
    return out

# ---------- App ----------
app = pg.mkQApp("Delta Analysis Heatmaps")

# Window 1: Combined
win_combined = pg.GraphicsLayoutWidget(title="Delta Analysis - Combined")
win_combined.setBackground("white")
p_combined = win_combined.addPlot(title="Combined State (0..15)")
p_combined.showGrid(x=True, y=True, alpha=0.15)

img_combined = pg.ImageItem()
img_combined.setLookupTable(full_lut)
img_combined.setImage(heatmap_data, autoLevels=False)
img_combined.setRect(rect)
p_combined.addItem(img_combined)

ltp_values = instance.ltpdf[:rows_used, 0].astype(float)
valid_ltp = np.isfinite(ltp_values) & (ltp_values != 0)
x = np.arange(rows_used, dtype=float)[valid_ltp]
y = ltp_values[valid_ltp]
p_combined.plot(x, y, pen=pg.mkPen(color=(0, 0, 255), width=2), name="LTP")

p_combined.setLabel("left", "LTP")
p_combined.setLabel("bottom", "Tick Index")
p_combined.setYRange(min_ltp, max_ltp, padding=0)
win_combined.show()

# Window 2: 2x2 Components
win_components = pg.GraphicsLayoutWidget(title="Delta Analysis - Components (2x2)")
win_components.setBackground("white")

# Bit-consistent mapping from combined formula:
# hl_buy=8, lh_buy=4, hl_sell=2, lh_sell=1
component_specs = [
    ("hl_buy (0 / 8 / transparent)", hl_buy, 8),
    ("lh_buy (0 / 4 / transparent)", lh_buy, 4),
    ("hl_sell (0 / 2 / transparent)", hl_sell, 2),
    ("lh_sell (0 / 1 / transparent)", lh_sell, 1),
]

plots = []
for i, (title, arr, true_code) in enumerate(component_specs):
    row, col = divmod(i, 2)
    p = win_components.addPlot(row=row, col=col, title=title)
    p.showGrid(x=True, y=True, alpha=0.15)

    img = pg.ImageItem()
    img.setLookupTable(full_lut)
    img.setImage(make_component_map(arr, true_code), autoLevels=False)
    img.setRect(rect)
    p.addItem(img)

    p.setLabel("left", "LTP")
    p.setLabel("bottom", "Tick Index")
    p.setYRange(min_ltp, max_ltp, padding=0)
    plots.append(p)

# Sync all 4 component plots to first plot
master = plots[0]
for p in plots[1:]:
    p.setXLink(master)
    p.setYLink(master)

all_plots = [p_combined] + plots
_sync_guard = {"busy": False}

def _sync_ranges(src_vb, ranges):
    if _sync_guard["busy"]:
        return
    _sync_guard["busy"] = True
    try:
        (x0, x1), (y0, y1) = ranges
        for p in all_plots:
            vb = p.getViewBox()
            if vb is src_vb:
                continue
            vb.setXRange(x0, x1, padding=0)
            vb.setYRange(y0, y1, padding=0)
    finally:
        _sync_guard["busy"] = False

for p in all_plots:
    p.getViewBox().sigRangeChanged.connect(_sync_ranges)

win_components.show()
app.exec()

0

In [6]:
# Generate CSV Files for each component
output_dir = f"{FILEPATH}/delta_components/{EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}"
os.makedirs(output_dir, exist_ok=True)
np.savetxt(f"{output_dir}/hl_buy.csv", hl_buy, delimiter=",", fmt="%s")
np.savetxt(f"{output_dir}/lh_buy.csv", lh_buy, delimiter=",", fmt="%s")
np.savetxt(f"{output_dir}/hl_sell.csv", hl_sell, delimiter=",", fmt="%s")
np.savetxt(f"{output_dir}/lh_sell.csv", lh_sell, delimiter=",", fmt="%s")   

# Old Versions- don't bother with these :D

In [ ]:
import numpy as np
import pyqtgraph as pg
from PyQt5.QtCore import QRectF

def get_active_price_bounds_local(inst):
    active = np.where((inst.aggdf_buy > 0) & (inst.aggdf_sell > 0))[0]
    if active.size == 0:
        return 0, max(0, inst.WIDTH - 1)
    return int(active[0]), int(active[-1])

# ---------- Build heatmap data from Delta_analysis numpy arrays ----------
rows_used = int(instance.curr_time_idx)
if rows_used <= 0:
    raise ValueError("No parsed ticks available in instance.")

min_idx, max_idx = get_active_price_bounds_local(instance)
col_slice = slice(min_idx, max_idx + 1)

# 1. Extract matrices
lh_buy = instance.lowHigh['buy'][:rows_used, col_slice]
hl_buy = instance.highLow['buy'][:rows_used, col_slice]
lh_sell = instance.lowHigh['sell'][:rows_used, col_slice]
hl_sell = instance.highLow['sell'][:rows_used, col_slice]

# 2. Find where data is valid (Not NaN)
valid_buy = ~np.isnan(lh_buy)
valid_sell = ~np.isnan(lh_sell)

# 3. Calculate codes cleanly as integers
buy_code = np.zeros_like(lh_buy, dtype=np.uint8)
sell_code = np.zeros_like(lh_sell, dtype=np.uint8)

buy_code[valid_buy] = (
    (lh_buy[valid_buy] >= 0).astype(np.uint8)
    + 2 * (hl_buy[valid_buy] >= 0).astype(np.uint8)
)

sell_code[valid_sell] = (
    (lh_sell[valid_sell] >= 0).astype(np.uint8)
    + 2 * (hl_sell[valid_sell] >= 0).astype(np.uint8)
)

# 4. Combine into strict integers (0 to 15)
heatmap_data = sell_code + 4 * buy_code

# 5. If EITHER buy or sell is missing, the pixel is transparent
invalid = ~(valid_buy & valid_sell)
heatmap_data[invalid] = 16

# ---------- PyQtGraph setup ----------
app = pg.mkQApp("Delta Analysis Heatmap")
win = pg.GraphicsLayoutWidget(title="Delta_analysis Heatmap & LTP")
win.setBackground("white")

# 17 Base Colors
base_lut = np.array([
    [255, 0, 0, 170],       # 0: Red
    [0, 114, 178, 170],     # 1: Blue
    [230, 159, 0, 170],     # 2: Orange
    [86, 180, 233, 170],    # 3: Sky Blue
    [0, 158, 115, 170],     # 4: Bluish Green
    [204, 121, 167, 170],   # 5: Purple
    [240, 228, 66, 170],    # 6: Yellow
    [0, 0, 128, 170],       # 7: Navy
    [255, 105, 180, 170],   # 8: Pink
    [27, 94, 32, 170],      # 9: Dark Green
    [139, 69, 19, 170],     # 10: Brown
    [128, 0, 0, 170],       # 11: Maroon
    [0, 128, 128, 170],     # 12: Teal
    [245, 245, 245, 170],   # 13: Off-White
    [119, 119, 119, 170],   # 14: Gray
    [57, 255, 20, 170],     # 15: Neon Green
    [0, 0, 0, 0],           # 16: TRANSPARENT (Background)
], dtype=np.ubyte)

# Pad LUT to exactly 256 rows so PyQtGraph does direct indexing
full_lut = np.zeros((256, 4), dtype=np.ubyte)
full_lut[:17] = base_lut

min_ltp = float(instance.base_ltp + min_idx)
max_ltp = float(instance.base_ltp + max_idx)
rect = QRectF(0, min_ltp, rows_used, max(1.0, max_ltp - min_ltp + 1.0))

# --- Plot 1: Combined ---
plot_combined = win.addPlot(title="Combined State (0..15)")
plot_combined.showGrid(x=True, y=True, alpha=0.15)
img_combined = pg.ImageItem()
img_combined.setLookupTable(full_lut)
img_combined.setImage(heatmap_data, autoLevels=False)
img_combined.setRect(rect)
plot_combined.addItem(img_combined)

ltp_values = instance.ltpdf[:rows_used, 0].astype(float)
valid_ltp = np.isfinite(ltp_values) & (ltp_values != 0)
x = np.arange(rows_used, dtype=float)[valid_ltp]
y = ltp_values[valid_ltp]
plot_combined.plot(x, y, pen=pg.mkPen(color=(0, 0, 255), width=2), name="LTP")
plot_combined.setLabel("left", "LTP")
plot_combined.setYRange(min_ltp, max_ltp, padding=0)

def make_component_map(arr, true_code):
    out = np.zeros_like(arr, dtype=np.uint8)  # default 0 => red
    valid = ~np.isnan(arr)
    out[valid] = np.where(arr[valid] >= 0, true_code, 0).astype(np.uint8)
    out[~valid] = 16  # transparent
    return out

# Bit-consistent mapping from combined formula:
# hl_buy=8, lh_buy=4, hl_sell=2, lh_sell=1
component_specs = [
    ("hl_buy (0 / 8 / transparent)", hl_buy, 8),
    ("lh_buy (0 / 4 / transparent)", lh_buy, 4),
    ("hl_sell (0 / 2 / transparent)", hl_sell, 2),
    ("lh_sell (0 / 1 / transparent)", lh_sell, 1),
]

for title, arr, true_code in component_specs:
    win.nextRow()
    p = win.addPlot(title=title)
    p.showGrid(x=True, y=True, alpha=0.15)
    p.setXLink(plot_combined)
    p.setYLink(plot_combined)
    img = pg.ImageItem()
    img.setLookupTable(full_lut)
    img.setImage(make_component_map(arr, true_code), autoLevels=False)
    img.setRect(rect)
    p.addItem(img)
    p.setLabel("left", "LTP")

plot_combined.setLabel("bottom", "Tick Index")
win.show()
app.exec()